In [33]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re

import warnings

warnings.filterwarnings("ignore", message=".*your specific warning message.*", category=UserWarning)


In [34]:
df = pd.read_csv(r"C:\Users\ranga\OneDrive\Desktop\BMW-Tech-Works\Task 3\dataset\Parts.csv", delimiter=';')

In [35]:
# dataset Info

print("dataset shape:", df.shape)
print("\nColumns and data Types:")
print(df.dtypes)
print("\nFirst 5 Rows of the dataset:")
df.head()

dataset shape: (998, 32)

Columns and data Types:
ID                                  object
DESCRIPTION                         object
Attribut1                           object
Additional Feature                  object
Application                         object
Characteristic                      object
Temp                                object
Height                              object
Length in mm                        object
Rating                              object
Material                            object
Size                                object
Code                                object
Joule-integral-Nom (J)              object
LC Risk                             object
Maximum AC Voltage Rating           object
Maximum DC Voltage Rating           object
Maximum Power Dissipation           object
Mounting                            object
Mounting Feature                    object
Number of Terminals                float64
Operating Temperature-Max (Cel)     object
Oper

,ID,DESCRIPTION,Attribut1,Additional Feature,Application,Characteristic,Temp,Height,Length in mm,Rating,...,Operating Temperature-Min (Cel),Physical Dimension,Pre-arcing time-Min (ms),Product Diameter,Product Length,Rated Breaking Capacity (A),Rated Current (A),Rated Voltage (V),Rated Voltage(AC) (V),Rated Voltage(DC) (V)
0,A1,Indicator Red Fast Movement 1.6A 250V Holder P...,Fast,NaN,Primary Protection In Equipment,VERY FAST,NaN,20mm,5.2mm,1.6A,...,-55Cel,5.2mm x 20mm,3ms,5.2mm,20mm,1500A,1.6A,250V,250V,NaN
1,A2,"Non Resettable Indicators Electric Indicator, ...",NaN,NaN,Primary Protection In Equipment,VERY FAST,NaN,20mm,5.2mm,NaN,...,-55Cel,5.2mm x 20mm,3ms,5.2mm,20mm,1500A,6.3A,250V,250V,NaN
2,A3,Indicator Red Fast Movement 8A 250V Holder Pla...,Fast,NaN,Primary Protection In Equipment,VERY FAST,NaN,20mm,5.2mm,8A,...,-55Cel,5.2mm x 20mm,10ms,5.2mm,20mm,1500A,8A,250V,250V,NaN
3,A4,"Non Resettable Indicators Electric Indicator, ...",NaN,NaN,Primary Protection In Equipment,VERY FAST,NaN,20mm,5.2mm,NaN,...,-55Cel,5.2mm x 20mm,10ms,5.2mm,20mm,1500A,10A,250V,250V,NaN
4,A5,Indicator Red Fast Movement 12.5A 250V Holder ...,Fast,NaN,Primary Protection In Equipment,VERY FAST,NaN,20mm,5.2mm,12.5A,...,-55Cel,5.2mm x 20mm,10ms,5.2mm,20mm,500A,12.5A,250V,250V,NaN


In [36]:
missing_counts = df.isna().sum()
print("Missing values per column:")
print(missing_counts)
print("\nPercentage of missing values per column:")
print((missing_counts / len(df)) * 100)

Missing values per column:
ID                                   0
DESCRIPTION                        335
Attribut1                          228
Additional Feature                 674
Application                        177
Characteristic                     326
Temp                               581
Height                             293
Length in mm                       293
Rating                             164
Material                           227
Size                               118
Code                               443
Joule-integral-Nom (J)             335
LC Risk                            212
Maximum AC Voltage Rating          264
Maximum DC Voltage Rating          550
Maximum Power Dissipation          745
Mounting                           224
Mounting Feature                   319
Number of Terminals                240
Operating Temperature-Max (Cel)    291
Operating Temperature-Min (Cel)    296
Physical Dimension                 330
Pre-arcing time-Min (ms)           89

## Observations and Data imputation statergy 
    - For the analytics DESCRIPTION is key feature to find similar parts. 
    - DESCRIPTION feature has lot on NAN values, dropping the columns will also have significant reduction in data size.
    - Filling DESCRIPTION by using mode could introduce Bias and every observation is unique.
    - Along with DESCRIPTION we have other parameter fields, we can use those to fill up DESCRIPTION since it resembles similar information.


In [37]:
# Filling description from other columns
def create_description(row):
    desc = row.get("DESCRIPTION", "")
    # Checking if DESCRIPTION is missing or empty
    if pd.isna(desc) or str(desc).strip() == "":
        cols_to_join = [col for col in row.index if col not in ['ID', 'DESCRIPTION']]  # Exclude ID and DESCRIPTION 
        values = []
        for col in cols_to_join:
            val = row[col]
            if pd.notna(val) and str(val).strip() != "":
                values.append(str(val).strip())
        new_desc = ", ".join(values)
        return new_desc if new_desc != "" else np.nan
    else:
        return desc

df['new_DESCRIPTION'] = df.apply(create_description, axis=1)

# Dropping rows if description is still empty
before_drop = len(df)
df = df[df['new_DESCRIPTION'].notna() & (df['new_DESCRIPTION'].str.strip() != "")]
after_drop = len(df)
print(f"Dropped {before_drop - after_drop} rows that contributed no description information.")

# Update the DESCRIPTION column with the new data
df['DESCRIPTION'] = df['new_DESCRIPTION']
df.drop(columns=['new_DESCRIPTION'], inplace=True)


print(len(df))
df.head()



Dropped 6 rows that contributed no description information.
992


,ID,DESCRIPTION,Attribut1,Additional Feature,Application,Characteristic,Temp,Height,Length in mm,Rating,...,Operating Temperature-Min (Cel),Physical Dimension,Pre-arcing time-Min (ms),Product Diameter,Product Length,Rated Breaking Capacity (A),Rated Current (A),Rated Voltage (V),Rated Voltage(AC) (V),Rated Voltage(DC) (V)
0,A1,Indicator Red Fast Movement 1.6A 250V Holder P...,Fast,NaN,Primary Protection In Equipment,VERY FAST,NaN,20mm,5.2mm,1.6A,...,-55Cel,5.2mm x 20mm,3ms,5.2mm,20mm,1500A,1.6A,250V,250V,NaN
1,A2,"Non Resettable Indicators Electric Indicator, ...",NaN,NaN,Primary Protection In Equipment,VERY FAST,NaN,20mm,5.2mm,NaN,...,-55Cel,5.2mm x 20mm,3ms,5.2mm,20mm,1500A,6.3A,250V,250V,NaN
2,A3,Indicator Red Fast Movement 8A 250V Holder Pla...,Fast,NaN,Primary Protection In Equipment,VERY FAST,NaN,20mm,5.2mm,8A,...,-55Cel,5.2mm x 20mm,10ms,5.2mm,20mm,1500A,8A,250V,250V,NaN
3,A4,"Non Resettable Indicators Electric Indicator, ...",NaN,NaN,Primary Protection In Equipment,VERY FAST,NaN,20mm,5.2mm,NaN,...,-55Cel,5.2mm x 20mm,10ms,5.2mm,20mm,1500A,10A,250V,250V,NaN
4,A5,Indicator Red Fast Movement 12.5A 250V Holder ...,Fast,NaN,Primary Protection In Equipment,VERY FAST,NaN,20mm,5.2mm,12.5A,...,-55Cel,5.2mm x 20mm,10ms,5.2mm,20mm,500A,12.5A,250V,250V,NaN


## Domain based filling approach for other features
- If we fill with traditional imputation techniques it will introduce bias ,so chosing to fill missing values with the available info in DESCRIPTION feature itself.
- if the info is not availble in description feature , we fill "NA" this way we maintain information bias balancing information loss.


Filling Attribut1

In [38]:
# Searching for patterns in the description column
attribute_patterns = [item for item in df['Attribut1'].unique() if not isinstance(item, float) or not item != item]

# Function to extract attribute type based on pattern matching
def extract_attribute(description):
    if pd.isna(description):
        return "NA"
    
    # Get text before first comma
    text_before_comma = description.split(',')[0] if ',' in description else description
    
    # Check for each pattern in the text
    for pattern in attribute_patterns:
        if re.search(r'\b' + re.escape(pattern) + r'\b', text_before_comma, re.IGNORECASE):
            return pattern
    
    return "NA"

df['Attribut1_Original'] = df['Attribut1']

mask = df['Attribut1'].isna() | (df['Attribut1'] == '')
df.loc[mask, 'Attribut1'] = df.loc[mask, 'DESCRIPTION'].apply(extract_attribute)

total_rows = len(df)
originally_filled = total_rows - sum(df['Attribut1_Original'].isna() | (df['Attribut1_Original'] == ''))
newly_filled_na = sum((df['Attribut1'] == 'NA') & (df['Attribut1_Original'].isna() | (df['Attribut1_Original'] == '')))


pattern_counts = {}
for pattern in attribute_patterns:
    pattern_counts[pattern] = sum((df['Attribut1'] == pattern) & 
                                 (df['Attribut1_Original'].isna() | (df['Attribut1_Original'] == '')))

total_newly_filled = sum(pattern_counts.values()) + newly_filled_na
total_filled_now = sum(~df['Attribut1'].isna() & (df['Attribut1'] != ''))
still_empty = total_rows - total_filled_now

# Statistics
print(f"Total rows in dataset: {total_rows}")
print(f"Originally filled Attribut1 values: {originally_filled}")
print("\nNewly filled attribute counts:")
for pattern, count in pattern_counts.items():
    print(f"  - '{pattern}': {count}")
print(f"  - 'NA' (non-matching): {newly_filled_na}")
print(f"\nTotal newly filled values: {total_newly_filled}")
print(f"Total filled values after processing: {total_filled_now}")
print(f"Still empty values: {still_empty}")
print(f"Percentage of dataset now filled: {total_filled_now/total_rows*100:.2f}%")

Total rows in dataset: 992
Originally filled Attribut1 values: 770

Newly filled attribute counts:
  - 'Fast': 12
  - 'Slow Blow': 1
  - 'Very Fast': 0
  - 'Medium Time Delay': 0
  - 'Super Time Lag': 0
  - 'NA' (non-matching): 209

Total newly filled values: 222
Total filled values after processing: 992
Still empty values: 0
Percentage of dataset now filled: 100.00%


Filling Additional Feature

In [39]:
def extract_additional_features(description):
    """
    Extract rated breaking capacity from description using pattern matching
    Returns formatted string if matches found, otherwise None
    """
    if pd.isna(description):
        return None
        
    # Spliting Description 
    parts = [p.strip() for p in description.split(',')]
    
    # Regular expression patterns
    voltage_pattern = r'(\d+\.?\d*)\s*(VAC|VDC)'
    current_pattern = r'(\d+\.?\d*)A\s*\(IR\)'
    
    # Stores matching values
    voltages = []
    breaking_current = None
      
    for part in parts:
        # Check for voltage matches
        voltage_match = re.search(voltage_pattern, part, re.IGNORECASE)
        if voltage_match:
            value, v_type = voltage_match.groups()
            voltages.append((v_type.upper(), f"{value} {v_type.upper()}"))
        
        # Check for breaking current
        current_match = re.search(current_pattern, part, re.IGNORECASE)
        if current_match:
            breaking_current = f"{current_match.group(1)} A"
    
    selected_voltage = None
    for v_type, voltage in reversed(voltages): 
        if v_type == 'VDC':
            selected_voltage = voltage
            break
    if not selected_voltage:
        for v_type, voltage in reversed(voltages):
            if v_type == 'VAC':
                selected_voltage = voltage
                break
       
    if selected_voltage and breaking_current:
        return f"RATED BREAKING CAPACITY AT {selected_voltage}: {breaking_current}"
    
    return "NA" 


# Applying function
original_empty = df['Additional Feature'].isna().sum()
df['New_Additional_Feature'] = df['DESCRIPTION'].apply(extract_additional_features)

# Update only empty original Additional Feature entries
update_mask = df['Additional Feature'].isna()
df.loc[update_mask, 'Additional Feature'] = df.loc[update_mask, 'New_Additional_Feature']
df = df.drop(columns=['New_Additional_Feature'])

# Handles remaining NA values
df['Additional Feature'].fillna("NA", inplace=True)

# statistics
filled_count = update_mask.sum()
successfully_filled = (df['Additional Feature'] != "NA").sum() - (original_empty - update_mask.sum())
remaining_empty = (df['Additional Feature'] == "NA").sum()

print(f"Filling Statistics:")
print(f"Total empty entries before: {original_empty}")
print(f"Successfully filled entries: {successfully_filled}")
print(f"NA entries: {remaining_empty}")
print(f"Success rate: {(successfully_filled/original_empty*100 if original_empty else 0):.2f}%")

# Filled samples
print("\nSample filled entry:")
filled_sample = df[update_mask].iloc[0:1]
print(filled_sample[['DESCRIPTION', 'Additional Feature']].to_string(index=False))


Filling Statistics:
Total empty entries before: 668
Successfully filled entries: 610
NA entries: 382
Success rate: 91.32%

Sample filled entry:
                                                                                                                                                                    DESCRIPTION                         Additional Feature
Indicator Red Fast Movement 1.6A 250V Holder Plastic 5 X 20mm Ceramic Box CCC/PSE/VDE/cULus Electric Indicator, Very Fast Blow, 1.6A, 250VAC, 1500A (IR), Inline/holder, 5x20mm RATED BREAKING CAPACITY AT 250 VAC: 1500 A


C:\Users\ranga\AppData\Local\Temp\ipykernel_12468\4030507622.py:59: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Additional Feature'].fillna("NA", inplace=True)


In [40]:
def fill_na_columns(df, columns):
 
    for col in columns:
        if col in df.columns:
            df[col] = df[col].fillna("NA")
    return df

# List of columns to fill with "NA" wherever there is a NaN value.
cols_to_fill = ["Application", "Material", "Code", "Joule-integral-Nom (J)","LC Risk","Mounting","Mounting Feature",
"Physical Dimension","Pre-arcing time-Min (ms)","Rated Breaking Capacity (A)",]

# Fill missing values for the specified columns.
df = fill_na_columns(df, cols_to_fill)


In [41]:
# Filling terminals with mean since the values are same for entire column
mean_terminals = df["Number of Terminals"].mean()
df["Number of Terminals"] = df["Number of Terminals"].fillna(mean_terminals)

In [42]:
# Searching patterns in Characteristic feature
characteristic_patterns = [item for item in df['Characteristic'].unique() 
                          if not isinstance(item, float) or not item != item]

# Modified function to check parts AFTER first chunk
def extract_characteristic(description):
    if pd.isna(description):
        return "NA"
    
    # Split into parts and exclude first chunk
    parts = description.split(',')
    text_after_first_comma = ','.join(parts[1:]) if len(parts) > 1 else ''
    
    # Check each pattern in remaining text
    for pattern in characteristic_patterns:
        if re.search(r'\b' + re.escape(str(pattern)) + r'\b', text_after_first_comma, re.IGNORECASE):
            return pattern
    
    return "NA"

df['Characteristic_Original'] = df['Characteristic']

mask = df['Characteristic'].isna() | (df['Characteristic'] == '')
df.loc[mask, 'Characteristic'] = df.loc[mask, 'DESCRIPTION'].apply(extract_characteristic)

total_rows = len(df)
originally_filled = total_rows - sum(df['Characteristic_Original'].isna() | (df['Characteristic_Original'] == ''))
newly_filled_na = sum((df['Characteristic'] == 'NA') & (df['Characteristic_Original'].isna() | (df['Characteristic_Original'] == '')))

# Calculate pattern counts
char_pattern_counts = {}
for pattern in characteristic_patterns:
    char_pattern_counts[pattern] = sum((df['Characteristic'] == pattern) & 
                                   (df['Characteristic_Original'].isna() | (df['Characteristic_Original'] == '')))

total_newly_filled = sum(char_pattern_counts.values()) + newly_filled_na
total_filled_now = sum(~df['Characteristic'].isna() & (df['Characteristic'] != ''))
still_empty = total_rows - total_filled_now

# statistics
print(f"Total rows in dataset: {total_rows}")
print(f"Originally filled Characteristic values: {originally_filled}")
print("\nNewly filled characteristic counts:")
for pattern, count in char_pattern_counts.items():
    print(f"  - '{pattern}': {count}")
print(f"  - 'NA' (non-matching): {newly_filled_na}")
print(f"\nTotal newly filled values: {total_newly_filled}")
print(f"Total filled values after processing: {total_filled_now}")
print(f"Still empty values: {still_empty}")
print(f"Percentage of dataset now filled: {total_filled_now/total_rows*100:.2f}%")

Total rows in dataset: 992
Originally filled Characteristic values: 672

Newly filled characteristic counts:
  - 'VERY FAST': 0
  - 'TIME LAG': 0
  - 'SUPER FAST': 0
  - 'FAST': 0
  - 'MEDIUM TIME LAG': 0
  - 'SLOW': 0
  - 'MEDIUM': 0
  - 'NA' (non-matching): 320

Total newly filled values: 320
Total filled values after processing: 992
Still empty values: 0
Percentage of dataset now filled: 100.00%


In [43]:
# Pattern for three numbers (Temp x Height x Length in mm)
pattern_three = re.compile(r'(\d+(?:\.\d+)?)\s*[xX×]\s*(\d+(?:\.\d+)?)\s*[xX×]\s*(\d+(?:\.\d+)?)\s*mm')
# Pattern for two numbers (Height x Length in mm)
pattern_two = re.compile(r'(\d+(?:\.\d+)?)\s*[xX×]\s*(\d+(?:\.\d+)?)\s*mm')

def extract_dimensions_from_description(desc):
    """
    Given a description string, try to extract dimensions.
    If a three-number pattern is found, return (Temp, Height, Length, "three").
    If only a two-number pattern is found, assume they represent Height and Length,
    and return ("NA", Height, Length, "two").
    If no pattern is found, return ("NA", "NA", "NA", "none").
    """
    if not isinstance(desc, str):
        return ("NA", "NA", "NA", "none")
    
    m_three = pattern_three.search(desc)
    if m_three:
        return m_three.group(1), m_three.group(2), m_three.group(3), "three"
    
    m_two = pattern_two.search(desc)
    if m_two:
        return "NA", m_two.group(1), m_two.group(2), "two"
    
    return ("NA", "NA", "NA", "none")


df['DESCRIPTION'] = df['DESCRIPTION'].fillna("")

# Applying the extraction function and expanding the result into new columns.
extraction_results = df['DESCRIPTION'].apply(extract_dimensions_from_description)
df[['Temp_extracted', 'Height_extracted', 'Length_in_mm_extracted', 'Extraction_type']] = pd.DataFrame(extraction_results.tolist(), index=df.index)


# Extraction Statistics
total_rows = len(df)
count_three = (df['Extraction_type'] == 'three').sum()
count_two   = (df['Extraction_type'] == 'two').sum()
count_none  = (df['Extraction_type'] == 'none').sum()

print("Total rows:", total_rows)
print("Rows filled by three-number pattern:", count_three, f"({(count_three/total_rows)*100:.1f}%)")
print("Rows filled by two-number pattern:", count_two, f"({(count_two/total_rows)*100:.1f}%)")


# For each target column (Temp, Height, Length in mm), fill missing values with extracted data.
for col, extracted_col in zip(["Temp", "Height", "Length in mm"],
                              ["Temp_extracted", "Height_extracted", "Length_in_mm_extracted"]):
    if col in df.columns:
        df[col] = df[col].fillna(df[extracted_col])
    else:
        df[col] = df[extracted_col]


df.drop(columns=["Temp_extracted", "Height_extracted", "Length_in_mm_extracted", "Extraction_type"], inplace=True)

# Display sample
print("\nSample of updated data with dimensions:")
print(df[["DESCRIPTION", "Temp", "Height", "Length in mm"]].head())


Total rows: 992
Rows filled by three-number pattern: 129 (13.0%)
Rows filled by two-number pattern: 353 (35.6%)

Sample of updated data with dimensions:
                                         DESCRIPTION Temp Height Length in mm
0  Indicator Red Fast Movement 1.6A 250V Holder P...   NA   20mm        5.2mm
1  Non Resettable Indicators Electric Indicator, ...   NA   20mm        5.2mm
2  Indicator Red Fast Movement 8A 250V Holder Pla...   NA   20mm        5.2mm
3  Non Resettable Indicators Electric Indicator, ...   NA   20mm        5.2mm
4  Indicator Red Fast Movement 12.5A 250V Holder ...   NA   20mm        5.2mm


In [44]:
def extract_electrical_specs(row):
    """Extract electrical specifications from the DESCRIPTION column."""
    desc = row['DESCRIPTION']
    
    def find_spec(unit):
        """Helper function: search for a number followed by the specified unit.
        Returns the matched string (e.g., '4W') or 'NA' if not found."""
        # Using re.escape(unit) ensures special characters are handled correctly.
        match = re.search(rf'\b(\d+(?:\.\d+)?)\s*{re.escape(unit)}\b', str(desc), re.IGNORECASE)
        return f"{match.group(1)}{unit}" if match else "NA"
    
    # Update each field only if its original value is missing (NaN or empty)
    if pd.isna(row['Rating']) or str(row['Rating']).strip() == "":
        row['Rating'] = find_spec("A")  # Current rating (e.g., "1A")
        
    if pd.isna(row['Maximum AC Voltage Rating']) or str(row['Maximum AC Voltage Rating']).strip() == "":
        row['Maximum AC Voltage Rating'] = find_spec("VAC")  # AC voltage (e.g., "250VAC")
        
    if pd.isna(row['Maximum DC Voltage Rating']) or str(row['Maximum DC Voltage Rating']).strip() == "":
        row['Maximum DC Voltage Rating'] = find_spec("VDC")  # DC voltage (e.g., "300VDC")
        
    if pd.isna(row['Maximum Power Dissipation']) or str(row['Maximum Power Dissipation']).strip() == "":
        row['Maximum Power Dissipation'] = find_spec("W")  # Power Dissipation (e.g., "4W")
    
    return row


df['DESCRIPTION'] = df['DESCRIPTION'].fillna("")

# Track original missing counts for each field.
original_empty = {
    'Rating': df['Rating'].isna().sum() if 'Rating' in df.columns else df.shape[0],
    'AC Voltage': df['Maximum AC Voltage Rating'].isna().sum() if 'Maximum AC Voltage Rating' in df.columns else df.shape[0],
    'DC Voltage': df['Maximum DC Voltage Rating'].isna().sum() if 'Maximum DC Voltage Rating' in df.columns else df.shape[0],
    'Power Dissipation': df['Maximum Power Dissipation'].isna().sum() if 'Maximum Power Dissipation' in df.columns else df.shape[0]
}

# Create masks for originally empty values.
rating_mask = df['Rating'].isna() if 'Rating' in df.columns else pd.Series([True]*len(df))
ac_mask = df['Maximum AC Voltage Rating'].isna() if 'Maximum AC Voltage Rating' in df.columns else pd.Series([True]*len(df))
dc_mask = df['Maximum DC Voltage Rating'].isna() if 'Maximum DC Voltage Rating' in df.columns else pd.Series([True]*len(df))
power_mask = df['Maximum Power Dissipation'].isna() if 'Maximum Power Dissipation' in df.columns else pd.Series([True]*len(df))

# Process the DataFrame row-wise.
df = df.apply(extract_electrical_specs, axis=1)

# Calculating fill statistics for each field.
results = {
    'Rating': {
        'filled': df.loc[rating_mask, 'Rating'].ne('NA').sum(),
        'na_filled': df.loc[rating_mask, 'Rating'].eq('NA').sum()
    },
    'AC Voltage': {
        'filled': df.loc[ac_mask, 'Maximum AC Voltage Rating'].ne('NA').sum(),
        'na_filled': df.loc[ac_mask, 'Maximum AC Voltage Rating'].eq('NA').sum()
    },
    'DC Voltage': {
        'filled': df.loc[dc_mask, 'Maximum DC Voltage Rating'].ne('NA').sum(),
        'na_filled': df.loc[dc_mask, 'Maximum DC Voltage Rating'].eq('NA').sum()
    },
    'Power Dissipation': {
        'filled': df.loc[power_mask, 'Maximum Power Dissipation'].ne('NA').sum(),
        'na_filled': df.loc[power_mask, 'Maximum Power Dissipation'].eq('NA').sum()
    }
}

print("Electrical Specification Enhancement Report:")
for spec, col_name in zip(['Rating', 'AC Voltage', 'DC Voltage', 'Power Dissipation'],
                          ['Rating', 'Maximum AC Voltage Rating', 'Maximum DC Voltage Rating', 'Maximum Power Dissipation']):
    print(f"\n{spec} Column:")
    print(f"  Originally empty: {original_empty[spec]}")
    print(f"  Successfully filled: {results[spec]['filled']}")
    print(f"  Filled with 'NA': {results[spec]['na_filled']}")
    print(f"  Remaining empty: {df[col_name].isna().sum()}")

# Displaying sample rows 
print("\nSample Filled Values:")
filled_samples = df[
    df['Rating'].ne('NA') | 
    df['Maximum AC Voltage Rating'].ne('NA') | 
    df['Maximum DC Voltage Rating'].ne('NA') |
    df['Maximum Power Dissipation'].ne('NA')
].sample(3, random_state=42)
print(filled_samples[['DESCRIPTION', 'Rating', 'Maximum AC Voltage Rating', 'Maximum DC Voltage Rating', 'Maximum Power Dissipation']].to_string(index=False))

print("\nSample NA Filled Values:")
na_samples = df[
    (df['Rating'] == 'NA') |
    (df['Maximum AC Voltage Rating'] == 'NA') |
    (df['Maximum DC Voltage Rating'] == 'NA') |
    (df['Maximum Power Dissipation'] == 'NA')
].sample(3, random_state=42)



Electrical Specification Enhancement Report:

Rating Column:
  Originally empty: 158
  Successfully filled: 110
  Filled with 'NA': 48
  Remaining empty: 0

AC Voltage Column:
  Originally empty: 258
  Successfully filled: 43
  Filled with 'NA': 215
  Remaining empty: 0

DC Voltage Column:
  Originally empty: 544
  Successfully filled: 107
  Filled with 'NA': 437
  Remaining empty: 0

Power Dissipation Column:
  Originally empty: 739
  Successfully filled: 0
  Filled with 'NA': 739
  Remaining empty: 0

Sample Filled Values:
                                                                                                                                                                                                DESCRIPTION Rating Maximum AC Voltage Rating Maximum DC Voltage Rating Maximum Power Dissipation
                      Indicator Chip Very Fast Movement 0.062A 125V SMD Solder Pad 2410 Ceramic T/R CE/CSA/PSE/UL Electric Indicator, Very Fast Blow, 0.062A, 125VAC, 125VDC, 50A (I

In [45]:
# List of redundant columns to remove
cols_to_remove = [
    "Size",
    "Physical Dimension", 
    "Pre-arcing time-Min (ms)", 
    "Product Diameter",
    "Product Length", 
    "Operating Temperature-Max (Cel)", 
    "Operating Temperature-Min (Cel)",
    "Rated Current (A)", 
    "Rated Voltage(AC) (V)",
    "Rated Voltage(DC) (V)", 
    "Characteristic_Original",
    "Attribut1_Original",
    "Rated Voltage (V)"
    
]


df.drop(columns=cols_to_remove, inplace=True, errors='ignore')

print("Remaining columns:")
print(df.columns)


Remaining columns:
Index(['ID', 'DESCRIPTION', 'Attribut1', 'Additional Feature', 'Application',
       'Characteristic', 'Temp', 'Height', 'Length in mm', 'Rating',
       'Material', 'Code', 'Joule-integral-Nom (J)', 'LC Risk',
       'Maximum AC Voltage Rating', 'Maximum DC Voltage Rating',
       'Maximum Power Dissipation', 'Mounting', 'Mounting Feature',
       'Number of Terminals', 'Rated Breaking Capacity (A)'],
      dtype='object')


In [46]:
df.isna().sum()

ID                             0
DESCRIPTION                    0
Attribut1                      0
Additional Feature             0
Application                    0
Characteristic                 0
Temp                           0
Height                         0
Length in mm                   0
Rating                         0
Material                       0
Code                           0
Joule-integral-Nom (J)         0
LC Risk                        0
Maximum AC Voltage Rating      0
Maximum DC Voltage Rating      0
Maximum Power Dissipation      0
Mounting                       0
Mounting Feature               0
Number of Terminals            0
Rated Breaking Capacity (A)    0
dtype: int64

In [47]:
df.to_csv("Parts_processed.csv", index=False)

## Summary

- **Challenges**
  - Missing values in primary features
  - Incomplete information across dataset
  - Strategic decisions needed for handling missing data

- **Approach**
  - Utilized alternative parameters to complete description fields
  - Eliminated redundant information features
  - Extracted key technical specifications (AC/DC voltage, ratings, attributes, characteristics) by parsing available DESCRIPTION data
  - Prioritized retention of domain information over traditional missing data techniques

- **Key Insight**
  - Domain information preservation was critical for similarity analysis
  - This approach maintained data integrity rather than simply filling gaps with statistically derived values that would lack ground truth